# Kolmogorov flow with JAX in Hydrogym

A recent development of Hydrogym is the inclusion of differentiable solvers like ``JAX``. The idea is to leverage the differentiable environment to compute sensitivity studies that can lead to a better controller with less compute time. This tutorial covers how to set up the Kolmogorov JAX environment and how to run basic control; reinforcement learning on the environment is shown in `../3_ppo/run_ppo.py`. Currently, the Kolmogorov flow is the main differentiable flow environment implemented in Hydrogym. For more details on the mathematical background of the Kolmogorov flow, please refer to the L4DC paper [1]. 

## 1. Running the environment 

In [ ]:
import os
import time
import jax.numpy as jnp
import jax
import matplotlib.pyplot as plt
import seaborn as sns
from hydrogym.jax.solvers.base import RungeKuttaCrankNicolson
from hydrogym.jax.utils import io as io
from hydrogym.jax.envs.kolmogorov import FlowConfig, PseudoSpectralNavierStokes2D

### Include postprocessing to monitor results 

In [ ]:
print_fmt = "vel1: {0:0.3f}\t\t vel2: {1:0.3f}\t\t vel3: {2:0.3e}\t\t vel4: {3:0.3e}"


def log_postprocess(flow):
    """
    The default observation is the velocity at 64 equally spaced points along the domain.
    This postprocess function is computing the mean of the observations.
    """
    obs = flow.get_observations()
    mean_obs_time = jnp.mean(obs, axis=1)
    return mean_obs_time


output_dir = "kolmogorov_data"
np_file_name = "kolmogorov_trajectory"
gif_file_name = "kolmogorov"
os.makedirs(output_dir, exist_ok=True)

### Set up solver parameters and run 

Some notes: 
- If you would like to change the grid resolution, define it as an argument of FlowConfig, like ``FlowConfig(grid_size=(256, 256))``; the default is 64 x 64 
- If you would like to change Reynolds number, specifically to view extreme events, set ``flow.Re``to be between 40 - 80. The default Reynolds number is 200, a fully turbulent state.

In [ ]:
flow = FlowConfig()
flow.Re = 100
dt = 0.001
equation = PseudoSpectralNavierStokes2D(flow)
solver = RungeKuttaCrankNicolson(flow, dt, 1, equation)
end_time = 50  # This is in seconds!

callbacks = [
    io.LogCallback(
        postprocess=log_postprocess,
        interval=1,
        filename=f"{output_dir}/kolmogorov.dat",
        print_fmt=print_fmt,
    ),
]

######## Check what device JAX is using! #######
# If you have access to a GPU, this should run much faster and you can improve grid resolution, default is 64x64
# Otherwise, this still runs pretty fast.


def check_jax_device():
    device = jax.default_backend()

    print(f"JAX is using: {device}")


check_jax_device()
start = time.time()
final, trajectory = solver.solve(dt, flow, (0, end_time), callbacks, None, save_n=1)
end = time.time()
jnp.save(output_dir + "/" + np_file_name, trajectory)
print("Total time:", end - start)

### Visualize results 

In [ ]:
cols = 5
fig, axs = plt.subplots(1, cols, figsize=(15, 5))

simulation = jnp.load(output_dir + "/" + np_file_name + ".npy")

data = jnp.fft.irfftn(simulation, axes=(1, 2))

for i in range(cols):
    time = int(len(data) * (i / cols))
    axs[i].imshow(data[time], cmap="icefire", vmin=-8, vmax=8)
    axs[i].set_title("time {}".format(time))

plt.tight_layout()
plt.show()

### Optional: Generate a gif 

In [ ]:
import numpy as np
import imageio.v2 as imageio

frames = []
for i in range(data.shape[0]):
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.imshow(data[i], cmap="icefire")
    ax.axis("off")
    fig.canvas.draw()
    # buffer_rgba() replaces the removed tostring_rgb() (Matplotlib >=3.8)
    image = np.asarray(fig.canvas.buffer_rgba())[:, :, :3]
    frames.append(image)

    plt.close(fig)

imageio.mimsave(output_dir + "/" + gif_file_name + ".gif", frames, fps=10)

## 2. Adding simple control

To actuate the JAX solver, pass a `control_field` directly to `solver.solve(...)`. The control field is a tuple `(fx, fy)` of physical-space forcing arrays defined on the solver grid (shape `grid_size x grid_size` here), which the equation adds to the momentum equation at every timestep. For instance, for a sinusoidal forcing of wavenumber 4 and amplitude -0.6 in `x`:

In [ ]:
x, y = flow.load_mesh(name="")


def control_func(a, k, y):
    return (a * jnp.sin(k * y), jnp.zeros_like(y))


# Tuple (fx, fy) of physical-space forcing arrays on the solver grid
control_field = control_func(-0.6, 4, y)

final, trajectory = solver.solve(dt, flow, (0, end_time), callbacks, None, save_n=1, control_field=control_field)
jnp.save(output_dir + "/" + "kolmogorov_controlled", trajectory)

### Visualize controlled results

In [ ]:
cols = 5
fig, axs = plt.subplots(1, cols, figsize=(15, 5))

simulation = jnp.load(output_dir + "/" + "kolmogorov_controlled" + ".npy")

data = jnp.fft.irfftn(simulation, axes=(1, 2))

for i in range(cols):
    time = int(len(data) * (i / cols))
    axs[i].imshow(data[time], cmap="icefire", vmin=-8, vmax=8)
    axs[i].set_title("time {}".format(time))

plt.tight_layout()
plt.show()

# References

[1] Jared Callaham, Sajeda Mokbel, Samuel Ahnert, Kai Lagemann, Esther Lagemann,
Nikolaus Adams, Christian Lagemann, Ludger Paehler, and Steven Brunton. 2025.
HydroGym: A Reinforcement Learning Platform for Fluid Dynamics. Accepted
to the Conference on Learning for Dynamics and Control (L4DC).
